In [1]:
import numpy as np
import pandas as pd
from tabularepimdl.SimpleTransition import SimpleTransition
from pydantic import BaseModel, Field
from typing import Annotated
from tabularepimdl.Rule import Rule
from numba import njit, prange

import time, timeit
#from memory_profiler import memory_usage
import tracemalloc #trace memory allocation

In [2]:
n = 10_000_000 #data size
rate = 0.1

In [3]:
# Dummy Data
np.random.seed(3)

df = pd.DataFrame({
    'InfState': np.random.choice(['S', 'I', 'R'], size=n),
    'N': np.random.randint(1, 10, size=n)
})

#users need to put the correct column headers in order for an array, so the rule logic can select the correct column for filtering, grouping and calculation
#Numpy and Numba solution does not have ability to identify which column is current state or population N
arr = np.column_stack([
    df['InfState'].values,
    df['N'].values
]) 

#create numerics mapping for S, I, R so that numba can work
state_map = {'S': 0, 'I': 1, 'R': 2}
arr_numba = arr.copy()
arr_numba[:, 0] = [state_map[val] for val in arr[:, 0]]
arr_numba = arr_numba.astype(np.float64)

arr_numba_p = arr_numba.copy()
#arr_numba

#from_st and to_st strings can only be used for pandas and numpy but not numba solution
from_st = 'S'
to_st = 'I'

In [4]:
#Solution A: NumPy-Only Version with OOP Wrapper
# The state dataframe's column order has to be fixed because get_deltas() needs to locate the right column to do filtering and calculation.
class SimpleTransitionNumpy(Rule, BaseModel):
    column_idx: int   # index of the column to check state (e.g., 0 for InfState)
    from_st: str
    to_st: str
    rate: Annotated[float, Field(ge=0)]
    stochastic: bool = False

    def get_deltas(self, current_state: np.ndarray, dt: float = 1.0) -> np.ndarray:
        infstate_col = current_state[:, self.column_idx] #column_idx needs to be 0 pointing to the first column 'InfState'
        mask = infstate_col == self.from_st

        # Extract N column (assumed to be at -1 index), column N has to be at a fixed position in the current_state
        N = current_state[mask, -1].astype(float) #select all values in column N that are S folks
        
        if not self.stochastic:
            changed_N = -N * (1 - np.exp(-dt * self.rate))
        else:
            changed_N = -np.random.binomial(N.astype(int), 1 - np.exp(-dt * self.rate))

        deltas = current_state[mask].copy() #create a S folks only array
        deltas[:, -1] = changed_N #number of S folks decreased

        deltas_add = deltas.copy()
        deltas_add[:, self.column_idx] = self.to_st #I folks
        deltas_add[:, -1] = -changed_N #number of I folks increased

        return np.vstack([deltas, deltas_add])
    
    def to_yaml(): pass


In [5]:
#Solution B: Numba-Accelerated Version
@njit
def compute_deltas_numba(current_state, column_idx, from_st, to_st, rate, dt, stochastic):
    n_rows = current_state.shape[0] #detect the number of rows and columns in input array
    n_cols = current_state.shape[1]

    #Worst case: all rows meet from_st requirement and will be used for calculation
    results = np.empty((n_rows * 2, n_cols), dtype=np.float64)

    count = 0 #track the number of original rows
    #the column_idx, from_st, to_st have to be numerics in order to allow numba to work
    for i in range(n_rows):
        row = current_state[i]

        if row[column_idx] == from_st:
            N = row[-1]
            
            #calculate one single delta value
            if stochastic:
                delta = -np.random.binomial(int(N), 1 - np.exp(-dt * rate))
            else:
                delta = -N * (1 - np.exp(-dt * rate))

            new_row = row.copy() 
            new_row[-1] = delta
            results[count] = new_row #decreased S folks
            count += 1

            added_row = row.copy() 
            added_row[column_idx] = to_st 
            added_row[-1] = -delta 
            results[count] = added_row #increased I folks
            count += 1

    return results[:count]


class SimpleTransitionNumba(BaseModel):
    column_idx: int
    from_st: int   # numeric is required
    to_st: int     # numeric is required
    rate: Annotated[float, Field(ge=0)]
    stochastic: bool = False

    def get_deltas(self, current_state: np.ndarray, dt: float = 1.0) -> np.ndarray:
        return compute_deltas_numba(
            current_state,
            self.column_idx,
            self.from_st,
            self.to_st,
            self.rate,
            dt,
            self.stochastic
        )


In [ ]:
simple_pandas = SimpleTransition(column='InfState', from_st=from_st, to_st=to_st, rate=rate)
simple_numpy = SimpleTransitionNumpy(column_idx=0, from_st=from_st, to_st=to_st, rate=rate)
simple_numba = SimpleTransitionNumba(column_idx=0, from_st=0, to_st=1, rate=rate)

In [ ]:
# Benchmarking functions
def run_pandas():
    simple_pandas.get_deltas(df)

def run_numpy():
    simple_numpy.get_deltas(arr)

def run_numba():
    simple_numba.get_deltas(arr_numba)


In [8]:
#Time Benchmarks
print("Execution time (1000 runs each)")

print("Pandas time (seconds):", round(timeit.timeit(run_pandas, number=1000), 2))
print("NumPy time (seconds):", round(timeit.timeit(run_numpy, number=1000), 2))
print("Numba time (seconds):", round(timeit.timeit(run_numba, number=1000), 2))

Execution time (1000 runs each)
Pandas time (seconds): 1513.82
NumPy time (seconds): 1507.94
Numba time (seconds): 918.18


In [9]:
#One-shot peak memory usage
tracemalloc.start()
run_pandas()
peak_mem_pandas = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

tracemalloc.start()
run_numpy()
peak_mem_numpy = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

tracemalloc.start()
run_numba()
peak_mem_numba = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

In [10]:
print("Peak memory usage (MiB) for one-shot run")

print(f"Pandas peak mem: {peak_mem_pandas/1024**2: .2f} MB")
print(f"NumPy peak mem: {peak_mem_numpy/1024**2: .2f} MB")
print(f"Numba peak mem: {peak_mem_numba/1024**2: .2f} MB")

Peak memory usage (MiB) for one-shot run
Pandas peak mem:  381.73 MB
NumPy peak mem:  416.70 MB
Numba peak mem:  305.18 MB


In [11]:
#Multiple runs to detect accumulation or leaks
print("Peak memory usage (MiB) for 100 runs")

tracemalloc.start()
for _ in range(100):
    run_pandas()
peak_mem_pandas = tracemalloc.get_traced_memory()[1]
print(f"Pandas 100 runs peak: {peak_mem_pandas/1024**2: .2f} MB")
tracemalloc.stop()

tracemalloc.start()
for _ in range(100):
    run_numpy()
peak_mem_numpy = tracemalloc.get_traced_memory()[1]
print(f"Numpy 100 runs peak: {peak_mem_numpy/1024**2: .2f} MB")
tracemalloc.stop()

tracemalloc.start()
for _ in range(100):
    run_numba()
peak_mem_numba = tracemalloc.get_traced_memory()[1]
print(f"Numba 100 runs peak: {peak_mem_numba/1024**2: .2f} MB")
tracemalloc.stop()

Peak memory usage (MiB) for 100 runs
Pandas 100 runs peak:  381.77 MB
Numpy 100 runs peak:  416.70 MB
Numba 100 runs peak:  305.18 MB


In [ ]:
n = 100_000 #data size

Execution time (1000 runs each)
Pandas time (seconds): 5.58
NumPy time (seconds): 3.71
Numba time (seconds): 7.7

Peak memory usage (MiB) for one-shot run
Pandas peak mem:  3.85 MB
NumPy peak mem:  4.19 MB
Numba peak mem:  3.05 MB

Peak memory usage (MiB) for 100 runs
Pandas 100 runs peak:  3.89 MB
Numpy 100 runs peak:  4.19 MB
Numba 100 runs peak:  3.06 MB

n = 1_000_000 #data size
Execution time (1000 runs each)
Pandas time (seconds): 124.53
NumPy time (seconds): 169.36
Numba time (seconds): 136.95

Peak memory usage (MiB) for one-shot run
Pandas peak mem:  38.21 MB
NumPy peak mem:  41.69 MB
Numba peak mem:  30.52 MB

Peak memory usage (MiB) for 100 runs
Pandas 100 runs peak:  38.25 MB
Numpy 100 runs peak:  41.70 MB
Numba 100 runs peak:  30.52 MB

n = 10_000_000 #data size
Execution time (1000 runs each)
Pandas time (seconds): 1513.82
NumPy time (seconds): 1507.94
Numba time (seconds): 918.18

Peak memory usage (MiB) for one-shot run
Pandas peak mem:  381.73 MB
NumPy peak mem:  416.70 MB
Numba peak mem:  305.18 MB

Peak memory usage (MiB) for 100 runs
Pandas 100 runs peak:  381.77 MB
Numpy 100 runs peak:  416.70 MB
Numba 100 runs peak:  305.18 MB
